# Batch 6 — E07, E08, E09, E15, E17 (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Runtime → Run all; about **2 h 30 min – 3 h** (E07 alone is about 1 h).
If Colab disconnects, Run all again: finished runs are skipped. E07 runs last so the cheaper runs finish first.

| Run | Change vs E21 (patch scales are "length, step"; E21 = 5,5 and 20,10) |
|---|---|
| E08 | Long scale 20,5 (9 pieces instead of 5) |
| E09 | Long scale 10,5 (11 pieces), matched to the 10-day horizon |
| E15 | One scale only: 16,8 (6 pieces) |
| E17 | 16 values per input instead of 64 after the patch branches; the next layer stays 128 wide (223,692 weights instead of 472,572) |
| E07 | Short scale 2,1 (59 pieces instead of 12); aimed at 1 day |

Rules and expectations: `claude/batches-4-7-design-2026-10-16.md`. Outputs in `MyDrive/Senior/results/batch6/`.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, os, subprocess, sys
for f in ['scripts/posthoc_eval.py', 'scripts/select_final.py', 'scripts/factors_tabular.py', 'src/alpha158.py', 'src/features_vol.py']:
    assert os.path.exists(f), f'stop: push the batch 4-7 code first ({f} missing)'
for f in ['scripts/baselines_deep.py', 'scripts/posthoc_eval.py', 'src/multiscale_patchtst.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
B3, B4, B5, B6, B7 = (f'{OUT_DIR}/batch{i}' for i in (3, 4, 5, 6, 7))
E21_JSON = f'{B3}/e21_stacked.json'
need = [E21_JSON, E21_JSON.replace('.json', '_preds.csv.gz'), f'{OUT_DIR}/xgb_e03.json', f'{OUT_DIR}/xgb_e03_preds.csv.gz']
need += [f'{B3}/e22/{f}_deep{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += [f'{OUT_DIR}/e05/{f}_xgb{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += []
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
for d in [B6]:
    os.makedirs(d, exist_ok=True)
print('earlier files found; batch 6 outputs go to', B6)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Tests; the notebook stops here if any fails (about 5 minutes)
for t in ['tests/test_pipeline.py', 'tests/test_train_options.py', 'tests/test_stopping.py', 'tests/test_multiscale_patchtst.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
E21 = ['--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
       '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10', '--cls-only', '--stop-rule', 'every',
       '--no-revin']
def done(path, n=3):
    try:
        return len(json.load(open(path))['models']['multiscale_sector']['runs']) == n
    except Exception:
        return False
def run(label, args, out, n=3):
    if done(out, n):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', *args, '--out', out], check=True)
def opt(args, **kw):
    a = list(args)
    for k, v in kw.items():
        k = '--' + k.replace('_', '-')
        if k in a: a[a.index(k) + 1] = str(v)
        else: a += [k, str(v)]
    return a

In [ ]:
run('E08 long scale 20,5', opt(E21, patches='5,5;20,5'), f'{B6}/e08_long_20_5.json')

In [ ]:
run('E09 long scale 10,5', opt(E21, patches='5,5;10,5'), f'{B6}/e09_long_10_5.json')

In [ ]:
run('E15 one scale 16,8', opt(E21, patches='16,8'), f'{B6}/e15_one_scale.json')

In [ ]:
run('E17 16 values per input', opt(E21, d_channel=16, trunk_width=128), f'{B6}/e17_d_channel16.json')

In [ ]:
run('E07 short scale 2,1', opt(E21, patches='2,1;20,10'), f'{B6}/e07_short_2_1.json')

In [ ]:
# Summary against E21 (per-horizon best epoch) -> batch6_summary.json (read by batch 7)
!python scripts/compare_batch.py --per-horizon-best --control-label E21 --control "$E21_JSON" \
    --run E07 "$B6/e07_short_2_1.json" --run E08 "$B6/e08_long_20_5.json" --run E09 "$B6/e09_long_10_5.json" \
    --run E15 "$B6/e15_one_scale.json" --run E17 "$B6/e17_d_channel16.json" --out "$B6/batch6_summary.json"